In [14]:
import sys
!{sys.executable} -m pip install --upgrade pyarrow

In [1]:
pip install python-dotenv loguru tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import sys
sys.path.append("../")

from olist.data import (
    load_raw_tables,
    clean_orders,
    clean_products,
    clean_geo,
    aggregate_payments,
    aggregate_items,
)

2026-05-24 19:40:42.145 | INFO     | olist.config:<module>:11 - PROJ_ROOT path is: D:\indie\Projects\ecommerce_analysis


## Load and clean each table:

In [2]:
tables = load_raw_tables()

orders_clean   = clean_orders(tables["orders"])
products_clean = clean_products(tables["products"], tables["cat_trans"])
geo_clean      = clean_geo(tables["geo"])
payments_agg   = aggregate_payments(tables["payments"])
items_agg      = aggregate_items(tables["items"])

print(f"Orders after cleaning:   {len(orders_clean):,}")
print(f"Products after cleaning: {len(products_clean):,}")
print(f"Payments aggregated:     {len(payments_agg):,}")
print(f"Items aggregated:        {len(items_agg):,}")

clean_orders: dropped 8 rows with null delivery dates
Orders after cleaning:   96,470
Products after cleaning: 32,951
Payments aggregated:     99,440
Items aggregated:        98,666


## Validation checks:

In [3]:
assert orders_clean["order_delivered_customer_date"].isnull().sum() == 0
assert orders_clean["order_estimated_delivery_date"].isnull().sum() == 0
assert orders_clean["order_id"].duplicated().sum() == 0
assert payments_agg["order_id"].duplicated().sum() == 0
assert items_agg["order_id"].duplicated().sum() == 0

print("✓ All validation checks passed")

✓ All validation checks passed


In [4]:
from olist.data import build_master

master = build_master(
    orders=orders_clean,
    customers=tables["customers"],
    items_agg=items_agg,
    payments_agg=payments_agg,
    reviews=tables["reviews"],
    products=products_clean,
    sellers=tables["sellers"],
    geo=geo_clean,
)

print(f"Master shape: {master.shape}")
print(f"Columns: {list(master.columns)}")

Master shape: (96999, 30)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_state', 'customer_city', 'customer_zip_code_prefix', 'n_items', 'total_price', 'total_freight', 'seller_id', 'product_id', 'payment_value', 'payment_installments', 'payment_type', 'n_payment_types', 'review_score', 'review_comment_message', 'is_bad_review', 'seller_state', 'seller_zip_code_prefix', 'product_category_name_english', 'product_weight_g', 'customer_lat', 'customer_lng', 'state']


## Null check:

In [5]:
null_pct = (master.isnull().sum() / len(master) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print("Remaining nulls after merge (%):")
print(null_pct.to_string())

Remaining nulls after merge (%):
review_comment_message    59.70
review_score               0.67
is_bad_review              0.67
customer_lat               0.28
state                      0.28
customer_lng               0.28
product_weight_g           0.02
order_approved_at          0.01


In [6]:
import importlib
import olist.data
importlib.reload(olist.data)
from olist.data import save_master

save_master(master)

Saved: D:\indie\Projects\ecommerce_analysis\data\interim\olist_master.parquet  (15.1 MB)


In [7]:
master_reload = pd.read_parquet("../data/interim/olist_master.parquet")
assert master_reload.shape == master.shape
print(f"✓ Reload verified — shape: {master_reload.shape}")
print(master_reload.dtypes)

✓ Reload verified — shape: (96999, 30)
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
customer_state                           object
customer_city                            object
customer_zip_code_prefix                  int64
n_items                                   int64
total_price                             float64
total_freight                           float64
seller_id                                object
product_id                               object
payment_value                           float64
payment_installments                    float64
payment_type                             object
n_payment_types                         float64
r